# Interactive simulation checks: effect propagation

Compares baseline vs the `mms_total_scaleup` scenario (common random numbers) to verify that
the oral-iron intervention *propagates*: MMS raises the hemoglobin and gestational-age
exposures, and that higher hemoglobin in turn lowers the hemoglobin->maternal-hemorrhage
relative risk and the maternal-hemorrhage incidence risk. Ported from the research portfolio
VnV notebook `model_18.3_interactive_simulation_effect_propogation`; updated to the current
Engine (`vivarium.engine`) API and to current model behavior.

Note: the source asserted MMS leaves the state-table hemoglobin and hemorrhage risk *unchanged*
(the effect being 'pending' in a separate pipeline). In the current model there is no such
split -- `hemoglobin.exposure` (pipeline) and `hemoglobin_exposure` (state column) coincide and
the effect propagates directly -- so the checks were rewritten to verify that propagation.

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [ ]:
!pip list | grep vivarium

In [ ]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
COLS = ["anc_attendance", "oral_iron_intervention", "age", "maternal_hemorrhage",
        "pregnancy_outcome", "gestational_age.exposure"]
PIPELINES = ["maternal_hemorrhage.incidence_risk",
             "hemoglobin_on_maternal_hemorrhage.incidence_risk.relative_risk", "hemoglobin.exposure"]

def run_to_hemorrhage(scenario=None):
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    get_event_name = sim._builder.time.simulation_event_name()
    while get_event_name() != "maternal_hemorrhage":
        sim.step()
    sim.step()  # advance past maternal_hemorrhage
    return sim

def frame(sim):
    df = sim.get_population(COLS + PIPELINES)
    # GA birth exposure is a column of the combined LBWSG birth-exposure pipeline.
    df["gestational_age.birth_exposure"] = sim.get_population(
        "low_birth_weight_and_short_gestation.birth_exposure"
    )["gestational_age"]
    return df

In [ ]:
baseline = run_to_hemorrhage()
mms = run_to_hemorrhage("mms_total_scaleup")
comp = frame(baseline).merge(frame(mms), left_index=True, right_index=True, suffixes=["_baseline", "_mms"])
comp.head()

## MMS propagates upstream: higher hemoglobin and gestational age

In [ ]:
# MMS (vs baseline, common random numbers) raises the hemoglobin and gestational-age exposures.
# In the current model the intervention effect is written into the state-table hemoglobin, so
# `hemoglobin.exposure` (pipeline) and `hemoglobin_exposure` (state column) coincide.
assert comp["hemoglobin.exposure_mms"].mean() > comp["hemoglobin.exposure_baseline"].mean(), \
    "MMS did not raise hemoglobin"
assert comp["gestational_age.exposure_mms"].mean() > comp["gestational_age.exposure_baseline"].mean(), \
    "MMS did not raise gestational-age exposure"
assert comp["gestational_age.birth_exposure_mms"].mean() > comp["gestational_age.birth_exposure_baseline"].mean(), \
    "MMS did not raise the gestational-age birth exposure"

## ...which propagates downstream to lower maternal-hemorrhage risk

In [ ]:
# Higher hemoglobin lowers the hemoglobin->maternal-hemorrhage relative risk, and hence the
# maternal-hemorrhage incidence risk.
assert comp["hemoglobin_on_maternal_hemorrhage.incidence_risk.relative_risk_mms"].mean() \
    < comp["hemoglobin_on_maternal_hemorrhage.incidence_risk.relative_risk_baseline"].mean(), \
    "MMS did not lower the hemoglobin->hemorrhage relative risk"
assert comp["maternal_hemorrhage.incidence_risk_mms"].mean() \
    < comp["maternal_hemorrhage.incidence_risk_baseline"].mean(), \
    "MMS did not lower maternal-hemorrhage incidence risk"

## Newly-covered simulants gain gestational age

In [ ]:
# Simulants switching from no treatment (baseline) to MMS gain gestational age. (Exact
# magnitude vs the artifact excess-shift is a good tightening for researchers to add.)
switchers = comp[(comp.oral_iron_intervention_baseline == "no_treatment")
                 & (comp.oral_iron_intervention_mms == "mms")]
observed_shift = (switchers["gestational_age.birth_exposure_mms"]
                  - switchers["gestational_age.birth_exposure_baseline"]).mean()
assert observed_shift > 0, \
    f"no_treatment->MMS switchers did not gain gestational age (shift={observed_shift:.3f})"

## Preterm birth is reduced by oral iron

In [ ]:
# Among ANC attendees, oral iron (IFA at baseline, MMS in the scenario) should reduce the
# preterm-birth rate relative to no treatment (relative risk < 1).
comp["preterm_baseline"] = comp["gestational_age.birth_exposure_baseline"] < 37
comp["preterm_mms"] = comp["gestational_age.birth_exposure_mms"] < 37
none_mask = (comp.oral_iron_intervention_baseline == "no_treatment") & (comp.anc_attendance_baseline != "none")
preterm_none = comp.loc[none_mask, "preterm_baseline"].mean()
preterm_ifa = comp.loc[comp.oral_iron_intervention_baseline == "ifa", "preterm_baseline"].mean()
preterm_mms = comp.loc[comp.oral_iron_intervention_mms == "mms", "preterm_mms"].mean()
assert preterm_ifa / preterm_none < 1.0, \
    f"IFA preterm RR {preterm_ifa / preterm_none:.3f} not protective (< 1)"
assert preterm_mms / preterm_ifa < 1.0, \
    f"MMS-vs-IFA preterm RR {preterm_mms / preterm_ifa:.3f} not protective (< 1)"